In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import trainval_transforms, revert_normalization, revert_standardization
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir, transform=trainval_transforms)

In [3]:
from src.model import Model
from torch.utils.data import DataLoader

trainval_dl = DataLoader(dataset, 8, True)
X_batch, y_batch = next(iter(trainval_dl))
model = Model()

model.eval()
with torch.no_grad():
    preds_batch = model(X_batch)

In [4]:
from src.postprocessing import postprocess_preds

postprocessed_preds = postprocess_preds(preds_batch[0])

In [9]:
postprocessed_preds

{'sofa': [(tensor(5.1820e-05),
   tensor(189.5896),
   tensor(32.0793),
   tensor(193.4923),
   tensor(31.6438)),
  (tensor(3.6761e-06),
   tensor(31.7994),
   tensor(1.0673),
   tensor(31.4269),
   tensor(0.4235))],
 'cow': [(tensor(0.0004),
   tensor(96.4132),
   tensor(97.7926),
   tensor(94.7029),
   tensor(94.7182)),
  (tensor(0.0004),
   tensor(126.2635),
   tensor(-0.8030),
   tensor(129.2661),
   tensor(0.8660))],
 'cat': [(tensor(0.0002),
   tensor(191.5419),
   tensor(2.2790),
   tensor(191.9896),
   tensor(-1.7953))],
 'bottle': [(tensor(0.0002),
   tensor(158.0064),
   tensor(193.3750),
   tensor(161.3676),
   tensor(191.8090)),
  (tensor(5.7062e-05),
   tensor(62.0529),
   tensor(31.1870),
   tensor(66.1195),
   tensor(32.3104))],
 'boat': [(tensor(2.5952e-05),
   tensor(128.4565),
   tensor(34.5287),
   tensor(127.4967),
   tensor(29.5461))],
 'motorbike': [(tensor(0.0002),
   tensor(-1.8405),
   tensor(63.8897),
   tensor(1.2997),
   tensor(64.1454)),
  (tensor(3.1600e-0

In [5]:
from src.utilities import objects_in_target

ground_truth_objects = objects_in_target(y_batch[0])

In [10]:
ground_truth_objects

{'person': [tensor([0.0625, 0.6250, 0.1741, 0.3080]),
  tensor([0.5312, 0.2812, 0.1384, 0.6116]),
  tensor([0.8750, 0.0000, 0.1562, 0.7098])]}

In [6]:
tp_fp_by_class = {
    "aeroplane": [],
    "bicycle": [],
    "bird": [],
    "boat": [],
    "bottle": [],
    "bus": [],
    "car": [],
    "cat": [],
    "chair": [],
    "cow": [],
    "diningtable": [],
    "dog": [],
    "horse": [],
    "motorbike": [],
    "person": [],
    "pottedplant": [],
    "sheep": [],
    "sofa": [],
    "train": [],
    "tvmonitor": [],
}

In [7]:
from src.evaluation import find_tp_and_fp

find_tp_and_fp(postprocessed_preds, ground_truth_objects, 
               tp_fp_by_class)

In [8]:
tp_fp_by_class

{'aeroplane': [(tensor(9.7084e-05), False)],
 'bicycle': [(tensor(0.0003), False),
  (tensor(0.0001), False),
  (tensor(5.9419e-05), False),
  (tensor(1.9789e-06), False)],
 'bird': [(tensor(9.7765e-05), False)],
 'boat': [(tensor(2.5952e-05), False)],
 'bottle': [(tensor(0.0002), False), (tensor(5.7062e-05), False)],
 'bus': [],
 'car': [],
 'cat': [(tensor(0.0002), False)],
 'chair': [],
 'cow': [(tensor(0.0004), False), (tensor(0.0004), False)],
 'diningtable': [],
 'dog': [(tensor(0.0003), False)],
 'horse': [(tensor(0.0002), False), (tensor(8.0167e-05), False)],
 'motorbike': [(tensor(0.0002), False), (tensor(3.1600e-05), False)],
 'person': [],
 'pottedplant': [],
 'sheep': [],
 'sofa': [(tensor(5.1820e-05), False), (tensor(3.6761e-06), False)],
 'train': [],
 'tvmonitor': [(tensor(0.0002), False)]}